# GLM-4.7-Flash STEM REAP Pruning

**REAP** (Router-weighted Expert Activation Pruning) for MoE models.

Based on [CerebrasResearch/reap](https://github.com/CerebrasResearch/reap):
```
S_j = (1/|X_j|) × Σ_{x∈X_j} g_j(x) · ||f_j(x)||_2
```

**Pipeline:**
- Base model: zai-org/GLM-4.7-Flash (64 experts)
- Target: 42 experts (33% pruning)
- Calibration: Siesher/mits-calibration-dataset
- Output: GGUF for LMStudio/llama.cpp

**Requirements:** Colab Pro+ with A100 40GB

## 1. GPU Check

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {gpu_mem:.1f} GB")
    if gpu_mem < 35:
        print(f"WARNING: A100 40GB recommended!")
else:
    raise RuntimeError("No CUDA GPU!")

name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB
PyTorch: 2.9.0+cu126
CUDA: True
GPU: NVIDIA A100-SXM4-40GB
Memory: 42.5 GB


## 2. Install Dependencies

In [2]:
!pip uninstall -y tensorflow tensorflow-cpu tf-keras -q 2>/dev/null || true
!pip install -q --upgrade pip
!pip install -q datasets huggingface_hub accelerate sentencepiece tqdm safetensors
!pip install -q git+https://github.com/huggingface/transformers.git

import transformers
print(f"[OK] transformers: {transformers.__version__}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 22.4 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
[OK] transformers: 5.0.1.dev0


In [3]:
!mkdir -p /content/models /content/outputs /content/gguf /content/offload
print("[OK] Directories created")

[OK] Directories created


In [4]:
from huggingface_hub import login
login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


## 3. Download Model

In [5]:
from huggingface_hub import snapshot_download
from transformers import AutoConfig
import os

MODEL_ID = "zai-org/GLM-4.7-Flash"
MODEL_PATH = "/content/models/glm-4.7-flash"

if not os.path.exists(f"{MODEL_PATH}/config.json"):
    print(f"Downloading {MODEL_ID}...")
    snapshot_download(
        repo_id=MODEL_ID,
        local_dir=MODEL_PATH,
        ignore_patterns=["*.gguf", "*.md", "*.txt"]
    )

config = AutoConfig.from_pretrained(MODEL_PATH, trust_remote_code=True)
print(f"\n[OK] Model: {config.model_type}")
print(f"Experts: {config.n_routed_experts}")
print(f"Layers: {config.num_hidden_layers}")
print(f"Active per token: {config.num_experts_per_tok}")

Fetching 57 files:   0%|          | 0/57 [00:00<?, ?it/s]


[OK] Model: glm4_moe_lite
Experts: 64
Layers: 47
Active per token: 4


## 4. REAP Pruning

In [6]:
# ============================================================================
# REAP PRUNING IMPLEMENTATION
# Fixed for GLM-4.7-Flash expert format (separate keys per expert)
# ============================================================================

import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig
from datasets import load_dataset
from tqdm import tqdm
import gc
import json
from safetensors.torch import save_file, load_file
import glob
import re

# ===== CONFIGURATION =====
OUTPUT_DIR = "/content/outputs/glm-stem-pruned"
MODEL_PATH = "/content/models/glm-4.7-flash"
DATASET_ID = "Siesher/mits-calibration-dataset"
OFFLOAD_FOLDER = "/content/offload"
COMPRESSION_RATIO = 0.33  # 33% pruning: 64 -> 42 experts
N_CALIBRATION_SAMPLES = 500

os.makedirs(OFFLOAD_FOLDER, exist_ok=True)


class REAPObserver:
    """
    REAP saliency observer for GLM4MoeLite.
    Computes: S_j = mean(g_j(x) * ||f_j(x)||_2) for selected experts.
    """
    
    def __init__(self, model, num_layers: int, num_experts: int, num_experts_per_tok: int):
        self.model = model
        self.num_layers = num_layers
        self.num_experts = num_experts
        self.num_experts_per_tok = num_experts_per_tok
        self.device = next(model.parameters()).device
        
        self.reap_sum = torch.zeros(num_layers, num_experts, device=self.device)
        self.reap_count = torch.zeros(num_layers, num_experts, device=self.device)
        self.hooks = []
        self.total_tokens = 0
    
    def _compute_expert_output_norm(self, hidden, experts, expert_idx):
        """Compute L2 norm of expert output."""
        gate_up = F.linear(hidden, experts.gate_up_proj[expert_idx])
        mid = gate_up.shape[-1] // 2
        activated = F.silu(gate_up[..., :mid]) * gate_up[..., mid:]
        output = F.linear(activated, experts.down_proj[expert_idx])
        return torch.linalg.norm(output, dim=-1)
    
    def _create_hook(self, layer_idx: int):
        def hook(module, args, output):
            hidden_states = args[0]
            if hidden_states.dim() == 2:
                hidden_states = hidden_states.unsqueeze(0)
            
            batch_size, seq_len, hidden_size = hidden_states.shape
            num_tokens = batch_size * seq_len
            self.total_tokens += num_tokens
            
            router_logits = module.gate(hidden_states)
            routing_weights = F.softmax(router_logits, dim=-1, dtype=torch.float32)
            topk_weights, topk_indices = torch.topk(routing_weights, self.num_experts_per_tok, dim=-1)
            
            hidden_flat = hidden_states.view(num_tokens, hidden_size)
            topk_indices_flat = topk_indices.view(num_tokens, self.num_experts_per_tok)
            routing_flat = routing_weights.view(num_tokens, self.num_experts)
            experts = module.experts
            
            with torch.no_grad():
                # Only iterate over unique selected experts (optimization)
                unique_experts = topk_indices_flat.unique()
                for expert_idx in unique_experts.tolist():
                    active_mask = (topk_indices_flat == expert_idx).any(dim=-1)
                    if not active_mask.any():
                        continue
                    
                    active_hidden = hidden_flat[active_mask]
                    active_weights = routing_flat[active_mask, expert_idx]
                    expert_norms = self._compute_expert_output_norm(active_hidden, experts, expert_idx)
                    
                    self.reap_sum[layer_idx, expert_idx] += (active_weights * expert_norms).sum()
                    self.reap_count[layer_idx, expert_idx] += active_mask.sum()
        return hook
    
    def register_hooks(self):
        for layer_idx in range(self.num_layers):
            layer = self.model.model.layers[layer_idx]
            if hasattr(layer.mlp, 'experts'):
                hook = layer.mlp.register_forward_hook(self._create_hook(layer_idx))
                self.hooks.append(hook)
        print(f"Registered {len(self.hooks)} hooks")
    
    def remove_hooks(self):
        for hook in self.hooks:
            hook.remove()
        self.hooks = []
    
    def get_global_saliency(self):
        """Compute global REAP saliency averaged across layers."""
        layer_means = self.reap_sum / self.reap_count.clamp(min=1)
        valid_layers = (self.reap_count.sum(dim=1) > 0)
        if valid_layers.sum() > 0:
            return layer_means[valid_layers].mean(dim=0)
        return layer_means.mean(dim=0)
    
    def get_experts_to_keep(self, target_experts: int):
        saliency = self.get_global_saliency()
        _, top_indices = torch.topk(saliency, target_experts)
        return sorted(top_indices.cpu().tolist())


def prune_and_save_from_files(model_path, experts_to_keep, target_experts, output_dir):
    """
    Prune experts stored as separate keys (experts.0, experts.1, etc.)
    and save with correct renumbering.
    """
    # Build mapping: old expert index -> new expert index
    experts_to_keep_sorted = sorted(experts_to_keep)
    old_to_new = {old_idx: new_idx for new_idx, old_idx in enumerate(experts_to_keep_sorted)}
    experts_set = set(experts_to_keep_sorted)
    
    print(f"Keeping {len(experts_to_keep_sorted)} experts: {experts_to_keep_sorted[:10]}...")
    
    os.makedirs(output_dir, exist_ok=True)
    
    # Update and save config
    config = AutoConfig.from_pretrained(model_path, trust_remote_code=True)
    config.n_routed_experts = target_experts
    config.save_pretrained(output_dir)
    
    # Copy tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    tokenizer.save_pretrained(output_dir)
    
    # Pattern to match expert keys: model.layers.X.mlp.experts.N.weight_name
    expert_pattern = re.compile(r'(model\.layers\.\d+\.mlp\.experts\.)(\d+)(\..*)')
    
    safetensor_files = sorted(glob.glob(os.path.join(model_path, "*.safetensors")))
    print(f"Processing {len(safetensor_files)} weight files...")
    
    pruned_state_dict = {}
    
    for sf_path in tqdm(safetensor_files, desc="Loading & pruning"):
        weights = load_file(sf_path)
        
        for key, value in weights.items():
            match = expert_pattern.match(key)
            
            if match:
                # Expert weight: model.layers.X.mlp.experts.N.weight
                prefix = match.group(1)
                expert_idx = int(match.group(2))
                suffix = match.group(3)
                
                if expert_idx in experts_set:
                    # Keep with new index
                    new_idx = old_to_new[expert_idx]
                    new_key = f"{prefix}{new_idx}{suffix}"
                    pruned_state_dict[new_key] = value
                # else: skip pruned expert
            
            elif '.mlp.gate.weight' in key:
                # Router weights: select rows for kept experts
                pruned_state_dict[key] = value[experts_to_keep_sorted].clone()
            
            elif '.mlp.gate.e_score_correction_bias' in key:
                # Router bias
                pruned_state_dict[key] = value[experts_to_keep_sorted].clone()
            
            else:
                # Keep as-is
                pruned_state_dict[key] = value
        
        del weights
        gc.collect()
    
    # Save with sharding (5GB per shard)
    print(f"Saving {len(pruned_state_dict)} tensors...")
    total_size = sum(t.numel() * t.element_size() for t in pruned_state_dict.values())
    max_shard_size = 5 * 1024 * 1024 * 1024
    
    if total_size <= max_shard_size:
        save_file(pruned_state_dict, os.path.join(output_dir, "model.safetensors"))
    else:
        current_shard, current_size, shard_idx = {}, 0, 1
        index = {"weight_map": {}, "metadata": {"total_size": total_size}}
        
        for key, tensor in tqdm(pruned_state_dict.items(), desc="Sharding"):
            tensor_size = tensor.numel() * tensor.element_size()
            if current_size + tensor_size > max_shard_size and current_shard:
                shard_name = f"model-{shard_idx:05d}-of-XXXXX.safetensors"
                save_file(current_shard, os.path.join(output_dir, shard_name))
                shard_idx += 1
                current_shard, current_size = {}, 0
            current_shard[key] = tensor
            current_size += tensor_size
            index["weight_map"][key] = f"model-{shard_idx:05d}-of-XXXXX.safetensors"
        
        if current_shard:
            save_file(current_shard, os.path.join(output_dir, f"model-{shard_idx:05d}-of-XXXXX.safetensors"))
        
        # Fix shard names
        total_shards = shard_idx
        for key in index["weight_map"]:
            index["weight_map"][key] = index["weight_map"][key].replace("XXXXX", f"{total_shards:05d}")
        
        for i in range(1, total_shards + 1):
            old = os.path.join(output_dir, f"model-{i:05d}-of-XXXXX.safetensors")
            new = os.path.join(output_dir, f"model-{i:05d}-of-{total_shards:05d}.safetensors")
            if os.path.exists(old):
                os.rename(old, new)
        
        with open(os.path.join(output_dir, "model.safetensors.index.json"), "w") as f:
            json.dump(index, f, indent=2)
    
    print(f"[OK] Saved to {output_dir}")


def run_reap_pruning():
    """Main REAP pruning pipeline."""
    print("="*60)
    print("GLM-4.7-Flash REAP Pruning")
    print("="*60)
    
    # Load config
    config = AutoConfig.from_pretrained(MODEL_PATH, trust_remote_code=True)
    num_experts = config.n_routed_experts
    num_layers = config.num_hidden_layers
    num_experts_per_tok = config.num_experts_per_tok
    target_experts = int(num_experts * (1 - COMPRESSION_RATIO))
    
    print(f"Experts: {num_experts} -> {target_experts} ({COMPRESSION_RATIO*100:.0f}% reduction)")
    
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    # Load model with offloading
    print("\nLoading model...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_PATH,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
        offload_folder=OFFLOAD_FOLDER
    )
    model.eval()
    
    # Load calibration dataset
    dataset = load_dataset(DATASET_ID, split="train")
    print(f"Dataset: {len(dataset)} samples")
    
    # Setup observer
    observer = REAPObserver(model, num_layers, num_experts, num_experts_per_tok)
    observer.register_hooks()
    
    # Calibration
    print(f"\nCalibrating on {N_CALIBRATION_SAMPLES} samples...")
    with torch.no_grad():
        for i in tqdm(range(min(N_CALIBRATION_SAMPLES, len(dataset)))):
            text = dataset[i]['instruction']
            if dataset[i].get('output'):
                text += "\n" + dataset[i]['output'][:500]
            inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
            try:
                model(**{k: v.to(model.device) for k, v in inputs.items()})
            except Exception as e:
                continue
    
    observer.remove_hooks()
    print(f"Tokens processed: {observer.total_tokens:,}")
    
    # Get experts to keep
    experts_to_keep = observer.get_experts_to_keep(target_experts)
    saliency = observer.get_global_saliency()
    print(f"\nSaliency range: [{saliency.min():.4f}, {saliency.max():.4f}]")
    
    # Free GPU memory before saving
    del model, observer
    gc.collect()
    torch.cuda.empty_cache()
    print("\nGPU memory freed.")
    
    # Prune and save
    print("\nPruning and saving...")
    prune_and_save_from_files(MODEL_PATH, experts_to_keep, target_experts, OUTPUT_DIR)
    
    # Save metadata
    metadata = {
        "base_model": "zai-org/GLM-4.7-Flash",
        "method": "REAP",
        "calibration_dataset": DATASET_ID,
        "calibration_samples": N_CALIBRATION_SAMPLES,
        "original_experts": num_experts,
        "pruned_experts": target_experts,
        "compression_ratio": COMPRESSION_RATIO,
        "experts_kept": experts_to_keep,
        "saliency_scores": saliency.cpu().tolist()
    }
    with open(f"{OUTPUT_DIR}/reap_metadata.json", "w") as f:
        json.dump(metadata, f, indent=2)
    
    print("\n" + "="*60)
    print("REAP Pruning Complete!")
    print("="*60)
    
    return experts_to_keep

In [7]:
# Clean previous run and start pruning
!rm -rf /content/outputs/glm-stem-pruned /content/offload

experts_kept = run_reap_pruning()

GLM-4.7-Flash REAP Pruning
Experts: 64 -> 42 (33% reduction)


`torch_dtype` is deprecated! Use `dtype` instead!



Loading model...


Loading weights:   0%|          | 0/751 [00:00<?, ?it/s]

Glm4MoeLiteForCausalLM LOAD REPORT from: /content/models/glm-4.7-flash
Key                                                 | Status     |  | 
----------------------------------------------------+------------+--+-
model.layers.47.self_attn.kv_a_layernorm.weight     | UNEXPECTED |  | 
model.layers.47.post_attention_layernorm.weight     | UNEXPECTED |  | 
model.layers.47.self_attn.o_proj.weight             | UNEXPECTED |  | 
model.layers.47.mlp.experts.gate_up_proj            | UNEXPECTED |  | 
model.layers.47.self_attn.kv_b_proj.weight          | UNEXPECTED |  | 
model.layers.47.mlp.gate.e_score_correction_bias    | UNEXPECTED |  | 
model.layers.47.input_layernorm.weight              | UNEXPECTED |  | 
model.layers.47.hnorm.weight                        | UNEXPECTED |  | 
model.layers.47.embed_tokens.weight                 | UNEXPECTED |  | 
model.layers.47.mlp.experts.down_proj               | UNEXPECTED |  | 
model.layers.47.enorm.weight                        | UNEXPECTED |  | 
model.

...

Tokens processed: 4,202,974

Saliency range: [nan, nan]

GPU memory freed.

Pruning and saving...
Keeping 42 experts: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]...
Processing 48 weight files...


Loading & pruning: 100%|██████████| 48/48 [00:18<00:00,  2.60it/s]


Saving 6601 tensors...


Sharding: 100%|██████████| 6601/6601 [03:42<00:00, 29.61it/s]


[OK] Saved to /content/outputs/glm-stem-pruned

REAP Pruning Complete!


## 5. Verification

**IMPORTANT:** Restart runtime before running verification to free GPU memory:
- VS Code: `Ctrl+Shift+P` → `Jupyter: Restart Kernel`
- Colab: `Runtime` → `Restart runtime`

In [1]:
# Verify pruned weights structure (no model loading needed)
from safetensors.torch import load_file
from transformers import AutoConfig
import glob
import os
import re

OUTPUT_DIR = "/content/outputs/glm-stem-pruned"

# Check config
config = AutoConfig.from_pretrained(OUTPUT_DIR, trust_remote_code=True)
print(f"Config n_routed_experts: {config.n_routed_experts}")

# Check actual weights
files = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.safetensors")))
print(f"Found {len(files)} safetensors files")

# Count unique experts in layer 1
expert_indices = set()
pattern = re.compile(r'model\.layers\.1\.mlp\.experts\.(\d+)\.')

for f in files:
    weights = load_file(f)
    for key in weights.keys():
        match = pattern.match(key)
        if match:
            expert_indices.add(int(match.group(1)))
    del weights

print(f"\nExperts in layer 1: {len(expert_indices)}")
print(f"Expert indices: {sorted(expert_indices)}")

# Verify gate weights
for f in files[:1]:
    weights = load_file(f)
    for key in weights.keys():
        if 'layers.1.mlp.gate.weight' in key:
            print(f"\ngate.weight shape: {weights[key].shape}")
            break

print("\n[OK] Structure verified!")

Config n_routed_experts: 42
Found 8 safetensors files

Experts in layer 1: 42
Expert indices: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41]

gate.weight shape: torch.Size([42, 2048])

[OK] Structure verified!


In [2]:
# Full model verification with generation test
# Run this AFTER restarting runtime

from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig
import torch
import os

OUTPUT_DIR = "/content/outputs/glm-stem-pruned"
OFFLOAD_FOLDER = "/content/offload_verify"
os.makedirs(OFFLOAD_FOLDER, exist_ok=True)

config = AutoConfig.from_pretrained(OUTPUT_DIR, trust_remote_code=True)
print(f"Loading pruned model ({config.n_routed_experts} experts)...")

model = AutoModelForCausalLM.from_pretrained(
    OUTPUT_DIR,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    trust_remote_code=True,
    offload_folder=OFFLOAD_FOLDER
)
tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR, trust_remote_code=True)

# Structure check
l1 = model.model.layers[1].mlp
print(f"\nlayer1.n_routed_experts: {l1.n_routed_experts}")
print(f"gate_up_proj.shape[0]: {l1.experts.gate_up_proj.shape[0]}")
print(f"gate.weight.shape[0]: {l1.gate.weight.shape[0]}")

# Generation test
print("\n" + "="*40)
print("Generation Test")
print("="*40)

prompts = [
    "Solve: 2x + 5 = 13",
    "Write a Python function to check if a number is prime",
    "Объясни закон Ньютона простыми словами"
]

model.eval()
for p in prompts:
    print(f"\n> {p}")
    inp = tokenizer(p, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inp, 
            max_new_tokens=100, 
            do_sample=False, 
            pad_token_id=tokenizer.eos_token_id
        )
    response = tokenizer.decode(out[0], skip_special_tokens=True)[len(p):].strip()
    print(response[:300])

print("\n[OK] Generation works!")

`torch_dtype` is deprecated! Use `dtype` instead!


Loading pruned model (42 experts)...


Loading weights:   0%|          | 0/751 [00:00<?, ?it/s]

Glm4MoeLiteForCausalLM LOAD REPORT from: /content/outputs/glm-stem-pruned
Key                                                 | Status     |  | 
----------------------------------------------------+------------+--+-
model.layers.47.mlp.shared_experts.down_proj.weight | UNEXPECTED |  | 
model.layers.47.shared_head.norm.weight             | UNEXPECTED |  | 
model.layers.47.self_attn.kv_a_proj_with_mqa.weight | UNEXPECTED |  | 
model.layers.47.mlp.experts.gate_up_proj            | UNEXPECTED |  | 
model.layers.47.self_attn.q_b_proj.weight           | UNEXPECTED |  | 
model.layers.47.self_attn.kv_b_proj.weight          | UNEXPECTED |  | 
model.layers.47.input_layernorm.weight              | UNEXPECTED |  | 
model.layers.47.mlp.gate.weight                     | UNEXPECTED |  | 
model.layers.47.enorm.weight                        | UNEXPECTED |  | 
model.layers.47.mlp.gate.e_score_correction_bias    | UNEXPECTED |  | 
model.layers.47.self_attn.q_a_layernorm.weight      | UNEXPECTED |  | 
mod


layer1.n_routed_experts: 42
gate_up_proj.shape[0]: 42
gate.weight.shape[0]: 42

Generation Test

> Solve: 2x + 5 = 13
Got it! The solution is 2x + 5 = 13. The answer is 4.

But I'm stuck on how to get there. Let's try to solve it step by step.

First, subtract 5 from both sides: 2x = 8.
Then, divide by 2: x = 4.

That's the solution.

Okay, I get it. The answer is 4.

But I'm stuck on how to get there. Let's

> Write a Python function to check if a number is prime
or not. The function should take a number as input and return True if the number is prime, else False. The function should be named is_prime.

We need to implement a function is_prime that checks if a number is prime. The function should take a number as input and return True if the number is prime,

> Объясни закон Ньютона простыми словами
. Закон Ньютона гласит,__(
1) 1. Сумма всех сил, действующих на объект, равна нулю, то объект не будет двигаться или изменять скорость. (2) Сумма всех сил, действующих на объект, не равна

In [3]:
# Clean up before GGUF conversion
try:
    del model
except:
    pass

import gc
import torch
gc.collect()
torch.cuda.empty_cache()
print("[OK] Memory cleared")

[OK] Memory cleared


## 6. GGUF Conversion

In [4]:
# Install llama.cpp
!git clone https://github.com/ggml-org/llama.cpp /content/llama.cpp 2>/dev/null || true
!pip install -q gguf numpy
print("[OK] llama.cpp ready")

[OK] llama.cpp ready


In [5]:
# Convert to GGUF FP16
import os
OUTPUT_DIR = "/content/outputs/glm-stem-pruned"
GGUF_FP16 = "/content/gguf/glm-stem-42exp-f16.gguf"

%cd /content/llama.cpp
!python convert_hf_to_gguf.py "{OUTPUT_DIR}" --outfile "{GGUF_FP16}" --outtype f16

if os.path.exists(GGUF_FP16):
    print(f"\n[OK] FP16: {os.path.getsize(GGUF_FP16)/1e9:.2f} GB")

/content/llama.cpp
INFO:hf-to-gguf:Loading model: glm-stem-pruned
INFO:hf-to-gguf:Model architecture: Glm4MoeLiteForCausalLM
INFO:hf-to-gguf:gguf: loading model weight map from 'model.safetensors.index.json'
INFO:hf-to-gguf:gguf: indexing model part 'model-00001-of-00008.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00002-of-00008.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00003-of-00008.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00004-of-00008.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00005-of-00008.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00006-of-00008.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00007-of-00008.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00008-of-00008.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,            torch.bfloat16 --> F16, sh

In [6]:
# Compile quantizer
%cd /content/llama.cpp
!make llama-quantize -j$(nproc) 2>/dev/null || echo "Already compiled or using pre-built"

/content/llama.cpp
Already compiled or using pre-built


In [7]:
# Quantize to Q4_K_M and Q8_0
import os
GGUF_FP16 = "/content/gguf/glm-stem-42exp-f16.gguf"
GGUF_Q4 = "/content/gguf/glm-stem-42exp-q4km.gguf"
GGUF_Q8 = "/content/gguf/glm-stem-42exp-q8.gguf"

%cd /content/llama.cpp

if os.path.exists(GGUF_FP16):
    !./llama-quantize "{GGUF_FP16}" "{GGUF_Q4}" Q4_K_M
    !./llama-quantize "{GGUF_FP16}" "{GGUF_Q8}" Q8_0
    
    print("\nGGUF files:")
    for f in [GGUF_FP16, GGUF_Q4, GGUF_Q8]:
        if os.path.exists(f):
            print(f"  {os.path.basename(f)}: {os.path.getsize(f)/1e9:.2f} GB")
    
    # Optionally remove FP16 to save space
    # os.remove(GGUF_FP16)
    print("\n[OK] Quantization complete!")
else:
    print("ERROR: FP16 GGUF not found. Run conversion first.")

/content/llama.cpp
/bin/bash: line 1: ./llama-quantize: No such file or directory
/bin/bash: line 1: ./llama-quantize: No such file or directory

GGUF files:
  glm-stem-42exp-f16.gguf: 40.80 GB

[OK] Quantization complete!


In [14]:
%cd /content/llama.cpp
                                                                                                                                                                                                                                        # Сборка с CUDA
!cmake -B build -DGGML_CUDA=ON                                                                                                                                                                                                          
!cmake --build build --config Release -j$(nproc) --target llama-quantize                                                                                                                                                                

# Проверка
!ls -la build/bin/llama-quantize 2>/dev/null || ls -la ./llama-quantize 2>/dev/null || echo "Checking build folder..."
!find . -name "llama-quantize" -type f 2>/dev/null



/content/llama.cpp
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
CMAKE_BUILD_TYPE=Release
-- Found Git: /usr/bin/git (found version "2.34.1")
-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU backend


In [16]:
import os
GGUF_FP16 = "/content/gguf/glm-stem-42exp-f16.gguf"
GGUF_Q4 = "/content/gguf/glm-stem-42exp-q4km.gguf"
GGUF_Q8 = "/content/gguf/glm-stem-42exp-q8.gguf"

# Путь к квантайзеру (после cmake build)
QUANTIZE = "/content/llama.cpp/build/bin/llama-quantize"

!"{QUANTIZE}" "{GGUF_FP16}" "{GGUF_Q4}" Q4_K_M
!"{QUANTIZE}" "{GGUF_FP16}" "{GGUF_Q8}" Q8_0

for f in [GGUF_Q4, GGUF_Q8]:
    if os.path.exists(f):
        print(f"✓ {os.path.basename(f)}: {os.path.getsize(f)/1e9:.2f} GB")

main: build = 7906 (1239267cc)
main: built with GNU 11.4.0 for Linux x86_64
main: quantizing '/content/gguf/glm-stem-42exp-f16.gguf' to '/content/gguf/glm-stem-42exp-q4km.gguf' as Q4_K_M
llama_model_loader: loaded meta data with 43 key-value pairs and 844 tensors from /content/gguf/glm-stem-42exp-f16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = deepseek2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                               general.name str              = Glm Stem Pruned
llama_model_loader: - kv   3:                         general.size_label str              = 42x2.6B
llama_model_loader: - kv   4:                      deepseek2.block_count u32              = 47
llama_model_loader: - kv   5:                   deepseek2.context

## 7. Upload to HuggingFace (Optional)

In [17]:
# Upload to HuggingFace

UPLOAD = True   
REPO = "Siesher/glm-stem-42exp-gguf"
                                                                                                                                                                                                                                        
if UPLOAD:
    from huggingface_hub import HfApi, create_repo
    import os

    api = HfApi()
    create_repo(REPO, exist_ok=True)

    files_to_upload = [
        "/content/gguf/glm-stem-42exp-q4km.gguf",
        "/content/gguf/glm-stem-42exp-q8.gguf",
        "/content/outputs/glm-stem-pruned/reap_metadata.json"
    ]

    for f in files_to_upload:
        if os.path.exists(f):
            print(f"Uploading {os.path.basename(f)}...")
            api.upload_file(
                path_or_fileobj=f,
                path_in_repo=os.path.basename(f),
                repo_id=REPO
            )

    print(f"\n[OK] https://huggingface.co/{REPO}")
else:
    print("[SKIP] Set UPLOAD=True to upload")

Uploading glm-stem-42exp-q4km.gguf...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  .../glm-stem-42exp-q4km.gguf:   0%|          | 7.25MB / 12.4GB            

Uploading glm-stem-42exp-q8.gguf...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...uf/glm-stem-42exp-q8.gguf:   0%|          | 9.36MB / 21.7GB            

No files have been modified since last commit. Skipping to prevent empty commit.


Uploading reap_metadata.json...

[OK] https://huggingface.co/Siesher/glm-stem-42exp-gguf


In [11]:
import os

                           
files = [
    "/content/gguf/glm-stem-42exp-q4km.gguf",                                                                                                                                                                                           
    "/content/gguf/glm-stem-42exp-q8.gguf",                                                                                                                                                                                             
    "/content/gguf/glm-stem-42exp-f16.gguf"
]

for f in files:
    if os.path.exists(f):
        print(f"✓ {os.path.basename(f)}: {os.path.getsize(f)/1e9:.2f} GB")
    else:
        print(f"✗ {os.path.basename(f)} - NOT FOUND")

✗ glm-stem-42exp-q4km.gguf - NOT FOUND
✗ glm-stem-42exp-q8.gguf - NOT FOUND
✓ glm-stem-42exp-f16.gguf: 40.80 GB


## 8. Summary

In [18]:
import os
import json

print("="*60)
print("GLM-4.7-Flash STEM REAP Pruning - COMPLETE")
print("="*60)

# Load metadata
try:
    with open("/content/outputs/glm-stem-pruned/reap_metadata.json") as f:
        m = json.load(f)
    print(f"\nMethod: {m['method']}")
    print(f"Experts: {m['original_experts']} -> {m['pruned_experts']}")
    print(f"Compression: {m['compression_ratio']*100:.0f}%")
    print(f"Calibration: {m['calibration_dataset']} ({m['calibration_samples']} samples)")
except Exception as e:
    print(f"Could not load metadata: {e}")

# List output files
print("\nOutput files:")
files = [
    "/content/outputs/glm-stem-pruned",
    "/content/gguf/glm-stem-42exp-f16.gguf",
    "/content/gguf/glm-stem-42exp-q4km.gguf",
    "/content/gguf/glm-stem-42exp-q8.gguf"
]
for f in files:
    if os.path.exists(f):
        if f.endswith(".gguf"):
            print(f"  {os.path.basename(f)}: {os.path.getsize(f)/1e9:.2f} GB")
        else:
            print(f"  HF model: {f}")

print("\n" + "="*60)
print("Usage:")
print("  LMStudio: Load .gguf file directly")
print("  Ollama:   ollama create glm-stem -f Modelfile")
print("  llama.cpp: ./llama-cli -m model.gguf -p 'prompt'")
print("="*60)

GLM-4.7-Flash STEM REAP Pruning - COMPLETE

Method: REAP
Experts: 64 -> 42
Compression: 33%
Calibration: Siesher/mits-calibration-dataset (500 samples)

Output files:
  HF model: /content/outputs/glm-stem-pruned
  glm-stem-42exp-f16.gguf: 40.80 GB
  glm-stem-42exp-q4km.gguf: 12.36 GB
  glm-stem-42exp-q8.gguf: 21.69 GB

Usage:
  LMStudio: Load .gguf file directly
  Ollama:   ollama create glm-stem -f Modelfile
  llama.cpp: ./llama-cli -m model.gguf -p 'prompt'


In [19]:
                                                                                                                                                                                                                                        
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import time

OUTPUT_DIR = "/content/outputs/glm-stem-pruned"

# Load model (skip if already loaded)
if 'model' not in dir() or model is None:
    print("Loading model...")
    model = AutoModelForCausalLM.from_pretrained(
        OUTPUT_DIR,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True
    )
    tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR, trust_remote_code=True)
    model.eval()

def generate(prompt, max_tokens=150):
    """Generate response with timing."""
    start = time.time()
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    elapsed = time.time() - start
    tokens_generated = outputs.shape[1] - inputs['input_ids'].shape[1]
    return response[len(prompt):].strip(), elapsed, tokens_generated

def test_category(name, prompts):
    """Test a category of prompts."""
    print(f"\n{'='*60}")
    print(f"  {name}")
    print('='*60)

    for prompt in prompts:
        print(f"\n📝 {prompt}")
        print("-" * 50)
        response, elapsed, tokens = generate(prompt)
        print(response[:500])
        print(f"\n⏱️ {elapsed:.1f}s | {tokens} tokens | {tokens/elapsed:.1f} tok/s")


# ============================================================================
# TEST CASES
# ============================================================================

# 1. ALGEBRA
test_category("ALGEBRA", [
    "Solve for x: 3x - 7 = 14",
    "Solve the quadratic equation: x² - 5x + 6 = 0",
    "Simplify: (2x + 3)(x - 4)",
    "Решите уравнение: 2x + 3 = 5x - 9",
])

# 2. CALCULUS
test_category("CALCULUS", [
    "Find the derivative of f(x) = x³ + 2x² - 5x + 1",
    "Calculate the integral: ∫ 2x dx",
    "Find the limit: lim(x→0) sin(x)/x",
    "Найди производную функции f(x) = e^x * sin(x)",
])

# 3. GEOMETRY
test_category("GEOMETRY", [
    "Find the area of a circle with radius 5",
    "Calculate the hypotenuse of a right triangle with legs 3 and 4",
    "What is the volume of a sphere with radius 3?",
    "Найди площадь треугольника со сторонами 5, 6, 7",
])

# 4. PHYSICS
test_category("PHYSICS", [
    "A car accelerates from 0 to 60 m/s in 10 seconds. What is the acceleration?",
    "Calculate the kinetic energy of a 2kg object moving at 5 m/s",
    "Explain Newton's second law of motion",
    "Чему равна сила притяжения между двумя телами массой 10 кг на расстоянии 1 м?",
])

# 5. PYTHON PROGRAMMING
test_category("PYTHON PROGRAMMING", [
    "Write a Python function to calculate factorial",
    "Write Python code to find all prime numbers up to 100",
    "Write a Python function to reverse a string",
    "Напиши функцию на Python для сортировки списка пузырьком",
])

# 6. ALGORITHMS
test_category("ALGORITHMS", [
    "Explain binary search algorithm",
    "What is the time complexity of quicksort?",
    "Write pseudocode for merge sort",
    "Объясни алгоритм поиска в ширину (BFS)",
])

# 7. DATA STRUCTURES
test_category("DATA STRUCTURES", [
    "Explain the difference between a stack and a queue",
    "How does a hash table work?",
    "What is a binary search tree?",
    "Объясни, что такое связный список",
])

# 8. CHEMISTRY
test_category("CHEMISTRY", [
    "Balance the equation: H2 + O2 → H2O",
    "What is the molecular weight of water (H2O)?",
    "Explain what happens during an acid-base neutralization",
    "Напиши электронную конфигурацию атома углерода",
])

# 9. WORD PROBLEMS
test_category("WORD PROBLEMS", [
    "A train travels 120 km in 2 hours. What is its average speed?",
    "If 5 workers can build a wall in 10 days, how long will 10 workers take?",
    "A store has a 20% discount. If an item costs $50, what is the final price?",
    "В магазине яблоки стоят 80 рублей за кг. Сколько стоит 2.5 кг?",
])

# 10. LOGIC & REASONING
test_category("LOGIC & REASONING", [
    "If all cats are animals, and all animals need food, what can we conclude about cats?",
    "What is the next number in the sequence: 2, 4, 8, 16, ?",
    "If A > B and B > C, what is the relationship between A and C?",
    "Продолжи последовательность: 1, 1, 2, 3, 5, 8, ?",
])

print("\n" + "="*60)
print("  TESTING COMPLETE")
print("="*60)


Loading model...


Loading weights:   0%|          | 0/751 [00:00<?, ?it/s]

KeyboardInterrupt: 